# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fmarryam70-ux/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [23]:
import os

if not os.path.exists("/content/flyrank-ml-internship"):
    !git clone https://github.com/fmarryam70-ux/flyrank-ml-internship.git /content/flyrank-ml-internship

os.chdir("/content/flyrank-ml-internship/work/notebooks")
print(os.getcwd())

/content/flyrank-ml-internship/work/notebooks


In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# label (starter proxy label, data mein already nahi hai — khud banani hai)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# ---- Signal check #1: staleness (flag-linked -> refresh flags iska use karte hain) ----
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)
sig1 = df.groupby("staleness_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
sig1.columns = ["decline_rate", "n"]
print("Signal 1: staleness vs decline rate")
print(sig1)
print("Verdict: CONFIRMED" if sig1["decline_rate"].iloc[-1] > sig1["decline_rate"].iloc[0] else "Verdict: check trend")

# ---- Signal check #2: CTR vs position (flag-linked -> CTR-fix logic iska use karta hai) ----
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-1, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"]
)
sig2 = df.groupby("position_bucket", observed=True)["ctr"].agg(["mean", "count"])
sig2.columns = ["avg_ctr", "n"]
print("\nSignal 2: position vs CTR")
print(sig2)
print("Verdict: CONFIRMED" if sig2["avg_ctr"].iloc[0] > sig2["avg_ctr"].iloc[-1] else "Verdict: check trend")

# Rule in plain words:
# "A page is worth reviewing if it's stale AND still gets visibility (staleness signal),
#  OR if it ranks well but has weak CTR (position/CTR signal) — both point to real recoverable opportunity."

Signal 1: staleness vs decline rate
                  decline_rate      n
staleness_bucket                     
<90d                  0.512031  20655
90-180d               0.611057   9171
180-365d              0.467456    169
365d+                 0.600000      5
Verdict: CONFIRMED

Signal 2: position vs CTR
                  avg_ctr      n
position_bucket                 
1-3              1.472869   2346
4-10             0.651045  11842
11-20            0.323443   7273
20+              0.211333   8539
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [25]:
import os

visible = (df["impressions_90d"] >= 500).astype(int)
stale = (df["days_since_last_update"] >= 180).astype(int)
low_ctr_visible = ((df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5) & visible).astype(int)

df["baseline_action_score"] = (
    0.55 * stale * visible * np.log1p(df["impressions_90d"])
    + 0.45 * low_ctr_visible * np.log1p(df["impressions_90d"])
)

conditions = [stale * visible == 1, low_ctr_visible == 1]
choices = ["stale_visible_page", "low_ctr_visible_page"]
df["reason_code"] = np.select(conditions, choices, default="no_signal")

df["action"] = np.where(df["baseline_action_score"] > 0, "review_for_refresh", "monitor")

queue = df.sort_values("baseline_action_score", ascending=False)[
    ["content_id", "baseline_action_score", "reason_code", "action", "is_declining_label"]
]

os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)
print(queue.head(10))

                 content_id  baseline_action_score         reason_code  \
16751  content_cf56e2e2e282              11.029699  stale_visible_page   
21268  content_0a91db491d14               9.495519  stale_visible_page   
12045  content_c2d929d83eaa               8.930494  stale_visible_page   
5327   content_fe16a55cd13d               8.424420  stale_visible_page   
20837  content_928af3e22c80               7.437206  stale_visible_page   
22872  content_e3ff1b093148               7.250636  stale_visible_page   
26840  content_7f116ae1f6f5               6.861711  stale_visible_page   
26799  content_77d4d5930e5e               6.720220  stale_visible_page   
7452   content_72496874f806               6.711740  stale_visible_page   
11630  content_6226ee6adc91               6.302619  stale_visible_page   

                   action  is_declining_label  
16751  review_for_refresh                   1  
21268  review_for_refresh                   1  
12045  review_for_refresh                

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [26]:
top10 = queue.head(10)
for i, row in top10.iterrows():
    print(f"content_id: {row['content_id']}")
    print(f"  action: {row['action']} | reason: {row['reason_code']} | score: {row['baseline_action_score']:.2f}")
    print(f"  why: flagged as '{row['reason_code']}' based on the rule above")
    print(f"  what would make it wrong: if this page's traffic drop is seasonal, not structural, or if it was already refreshed recently\n")

content_id: content_cf56e2e2e282
  action: review_for_refresh | reason: stale_visible_page | score: 11.03
  why: flagged as 'stale_visible_page' based on the rule above
  what would make it wrong: if this page's traffic drop is seasonal, not structural, or if it was already refreshed recently

content_id: content_0a91db491d14
  action: review_for_refresh | reason: stale_visible_page | score: 9.50
  why: flagged as 'stale_visible_page' based on the rule above
  what would make it wrong: if this page's traffic drop is seasonal, not structural, or if it was already refreshed recently

content_id: content_c2d929d83eaa
  action: review_for_refresh | reason: stale_visible_page | score: 8.93
  why: flagged as 'stale_visible_page' based on the rule above
  what would make it wrong: if this page's traffic drop is seasonal, not structural, or if it was already refreshed recently

content_id: content_fe16a55cd13d
  action: review_for_refresh | reason: stale_visible_page | score: 8.42
  why: flagg

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [27]:
weak = top10[top10["reason_code"] == "no_signal"]
print("Weak/questionable picks:", len(weak))

# leakage check
future_cols = [c for c in df.columns if "future" in c.lower() or "label" in c.lower()]
print("Leakage check — no future/label columns used as features:", future_cols)
print("Confirmed: only observable pre-decision signals used (staleness, CTR, position, impressions).")

Weak/questionable picks: 0
Leakage check — no future/label columns used as features: ['is_declining_label']
Confirmed: only observable pre-decision signals used (staleness, CTR, position, impressions).


In [28]:
print("Manual weak-pick note:")
print(f"Even the top pick ({top10.iloc[0]['content_id']}) could be a false positive if its")
print("traffic drop is seasonal (e.g. holiday content) rather than genuine content decay —")
print("the rule can't distinguish seasonal dips from structural decline.")

Manual weak-pick note:
Even the top pick (content_cf56e2e2e282) could be a false positive if its
traffic drop is seasonal (e.g. holiday content) rather than genuine content decay —
the rule can't distinguish seasonal dips from structural decline.


In [29]:
print("Note: reason_code priority means stale_visible_page can mask a co-occurring")
print("low_ctr_visible_page condition on the same row — this is a labeling choice, not an error.")

Note: reason_code priority means stale_visible_page can mask a co-occurring
low_ctr_visible_page condition on the same row — this is a labeling choice, not an error.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.